# Dairy Annual Report Scraping and Review

Extracts the six requested fields from 2024 dairy annual report PDFs that are absent from
`Dairy_Annual_Report_Summary_Table_Region_5_2024.csv`, then stages them for
manual review before they are written into that CSV.

Sources: the `Consulting_and_Error_Causing_Forms` and `Correct_Forms` folders under
`data/annual-reports/<office>/`.

Four form layouts are handled, each read by coordinate rather than by reading order:

| Layout | Detected by | Manure basis |
|---|---|---|
| Standard Water Board (R5-2007-0035) | `AVAILABLE NUTRIENTS` / `DAIRY FACILITY INFORMATION` | as-excreted |
| Innovative Ag Services | `INNOVATIVE AG SERVICES` | as-excreted |
| Livingston Dairy Consulting | `Livingston Dairy Consulting` | 40% moisture |
| Provost & Pritchard | `ppeng.com` | solid, as-is |

Herd counts come from the **Average number** row, which is what the existing 588 rows use
(verified against Avalon Dairy Farms and Maya Dairy).

Pipeline: `1` scrape -> `2` post-process -> `2b` crop N removal -> `3` build the review page -> `4` review in the browser -> `5` apply -> `5b` annual-report formats -> `6` coverage.

## How to run

Run this notebook from its own folder, `review/annual-reports/`.

The published files are already in place: `results.pkl`, `results_final.pkl`,
`crop_removal_extracted.csv`, `scraped_rows.csv`, the review page and `crops/` in this folder,
and `scrape_review.csv`, `Annual_Report_Formats.csv` and
`Dairy_Annual_Report_Summary_Table_Region_5_2024.csv` in
`outputs/review/annual-reports/`. Do not use Run All:

- Sections 1, 2, 2b and 3 rewrite the files in this folder. They read the PDFs in
  `data/annual-reports/` and use Apple Vision OCR, so they run on macOS only.
- Section 5 with `--write` and section 5b rewrite the files in `outputs/review/annual-reports/`.

Sections 2b, 3 and 5b read the scraping CSV that section 5 writes, so the published files come
from repeating 5 -> 5b -> 2b -> 3 -> 4 -> 5 until the review was complete.


In [ ]:
import os
import sys

# Run from review/annual-reports/: the scraper modules are in scraper/ and
# build_report_formats.py is in this folder.
NOTEBOOK_DIR = os.getcwd()
SCRAPER_DIR = os.path.join(NOTEBOOK_DIR, "scraper")
assert os.path.isdir(SCRAPER_DIR), "run this notebook from review/annual-reports/"
for _p in (SCRAPER_DIR, NOTEBOOK_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import run_extract
import finalize
import crop_removal
import build_review
import apply_rows
import coverage
import build_report_formats


## 1. Scrape

Reads each PDF listed in `list_targets.txt` from `data/annual-reports/`, reads the text layer where there is one and
falls back to Apple Vision OCR where there is not, then routes the document to the matching
form reader. Writes `results.pkl`.

Takes roughly 25 minutes for 360 reports.

In [ ]:
rows = run_extract.main()

## 2. Post-process

Three corrections on top of the raw extraction, all recorded in `Extraction Notes`:

- **Provost & Pritchard** rows switch to Attachment A's ASAE D384.2 whole-herd table. The
  "Solid Manure Generated" table covers only the separated solid stream, which reports about
  a twentieth of the herd's nitrogen; the ASAE table is whole-herd and its nitrogen is
  comparable with the standard form. Its tonnage is on a total-solids (dry) basis, so those
  rows carry `Manure Basis = asae-total-solids`.
- A **blank export figure becomes 0** where the form states no nutrient exports were entered.
- The **UCCE nitrogen** estimator and its ratio are filled. These come from herd counts alone,
  so they are independent of whatever moisture basis the form used - which is what makes the
  Livingston and Provost & Pritchard rows comparable with the rest. The per-category rates are
  refitted from the existing rows at run time and asserted to reproduce them exactly.

Writes `results_final.pkl`.

In [ ]:
finalize.main()

## 2b. Crop nitrogen removal

Reads the facility-wide crop nitrogen removal for every summary report filed on the standard
Water Board form ("Actual crop nutrient removal") or the Innovative Ag Services form ("Crop
Nitrogen Removal", Attachment B summary). This covers the reports parsed before this study as
well as those scraped here. Livingston reports print removal only per field and are left out.

Each value is checked against two figures printed on the same page. The ratio there is total
N applied from every source over crop removal, so a correctly read page satisfies
total / removal = ratio to within its two-decimal rounding. Pages the text layer cannot read
are OCR'd. Writes `crop_removal_extracted.csv` with a status per report and renders the page
into `crops/`.

In [ ]:
crop = crop_removal.main()

## 3. Build the review page

Grades the post-processed records in `results_final.pkl` and renders their evidence crops,
then writes `Dairy_Report_Scrape_Review.html`, `crops/`, `scraped_rows.csv` and
`triage_report.csv`. Decisions already in `scrape_review.csv` are carried into the page.


In [ ]:
build_review.main("results_final.pkl")


## 4. Review

Open `Dairy_Report_Scrape_Review.html`. Evidence crops sit on the left, the extracted values
on the right, and the identity checks re-evaluate as you type.

`1` accept · `2` flag · `3` unusable · `r` ratio ok · `c` crop N ok · `e` edit · arrow keys move

Press **Link CSV** once and pick `outputs/review/annual-reports/scrape_review.csv`; decisions are then written straight to
that file as you go, and reloading resumes where you left off. **Export decisions** downloads
the same file if you would rather not link it.

The queue is ordered failures first, then warnings, then OCR-read rows, then a 10% sample of
clean rows for spot checking.

The **Crop N** filter lists every facility-wide crop N removal not yet examined, problems
first: values whose page check fails or cannot run, pages not found, then zero removals, then
values that pass. Reports parsed before this study appear only under this filter, carrying
the crop removal and its page image and no verdict buttons. `c` confirms the value on screen;
typing a correction (or clearing the field when the page has no value) records it instead,
and the page check recomputes as you type. Both are written to the `crop_ok` and
`corrected_value` columns of `scrape_review.csv`.

If the browser blocks the file, serve the repository root and open
`http://localhost:8731/review/annual-reports/Dairy_Report_Scrape_Review.html`, so the
links to the PDFs in `data/annual-reports/` still resolve:

```bash
cd ../.. && python3 -m http.server 8731
```

## 5. Apply to the CSV

Appends the reviewed rows, applying any corrections and dropping anything marked unusable.
Keyed on `Source PDF`, so re-running after more review replaces rows rather than duplicating
them. Runs as a dry run until `write=True`.

Every rebuild also appends `Total Crop-Removed N` from `crop_removal_extracted.csv`. A value
corrected in review is written as corrected; otherwise the extracted value is written when its
page check passes or a reviewer confirmed it, and anything else stays blank. The run prints
how many values came from each route.

In [ ]:
import sys
sys.argv = ["apply_rows"]           # dry run: writes preview_merged.csv only
apply_rows.main()

In [ ]:
sys.argv = ["apply_rows", "--write"]
apply_rows.main()

## 5b. Annual-report formats

Tallies which form each report used (Figure S1a), linking every PDF in `data/annual-reports/`
to its row of the scraping CSV, and writes `outputs/review/annual-reports/Annual_Report_Formats.csv`.
Section 2b reads this file. The walk takes about twenty minutes.


In [ ]:
build_report_formats.main()


## 6. Coverage check

How much of the manifest export tonnage the summary now accounts for. Before the new rows,
dairies missing from the summary held 60.6% of the 2.33M wet tons in the manifests.

In [ ]:
coverage.coverage(f"{build_review.OUT}/backup_before_append.csv", "BEFORE")
coverage.coverage(apply_rows.TARGET, "AFTER")
